In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Load processed dataset
df = pd.read_csv("../data/processed/processed_ai4i.csv")

# Create features and target
X = df.drop(["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"], axis=1)
y = df["Machine failure"]

# Clean feature names for XGBoost (strip brackets '[' and ']')
X.columns = [col.replace("[", "").replace("]", "").strip() for col in X.columns]

X.head()

,Type,Air temperature K,Process temperature K,Rotational speed rpm,Torque Nm,Tool wear min
0,1,298.1,308.6,1551,42.8,0
1,0,298.2,308.7,1408,46.3,3
2,0,298.1,308.5,1498,49.4,5
3,0,298.2,308.6,1433,39.5,7
4,0,298.2,308.7,1408,40.0,9


In [4]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Compute scale_pos_weight for handling class imbalance in XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

scale_pos_weight: 28.52


In [5]:
# Initialize and train XGBoost Model
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)

# Initialize and train Weighted XGBoost Model
xgb_weighted = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)
xgb_weighted.fit(X_train, y_train)

print("XGBoost models trained successfully.")

XGBoost models trained successfully.


In [6]:
# Predict and evaluate Default XGBoost
y_pred_xgb = xgb_model.predict(X_test)
print("=== XGBoost Default Classification Report ===")
print(classification_report(y_test, y_pred_xgb))

# Predict and evaluate Weighted XGBoost
y_pred_weighted = xgb_weighted.predict(X_test)
print("=== XGBoost Weighted Classification Report ===")
print(classification_report(y_test, y_pred_weighted))

=== XGBoost Default Classification Report ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.87      0.69      0.77        68

    accuracy                           0.99      2000
   macro avg       0.93      0.84      0.88      2000
weighted avg       0.99      0.99      0.99      2000

=== XGBoost Weighted Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.75      0.78      0.76        68

    accuracy                           0.98      2000
   macro avg       0.87      0.89      0.88      2000
weighted avg       0.98      0.98      0.98      2000



In [7]:
# Confusion Matrix comparison
print("XGBoost Default Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nXGBoost Weighted Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_weighted))

XGBoost Default Confusion Matrix:
[[1925    7]
 [  21   47]]

XGBoost Weighted Confusion Matrix:
[[1914   18]
 [  15   53]]
